<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/Feedback_PID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 回授控制與 PID：開迴路／閉迴路、P、PI、PD、PID

對照網頁：[自動控制｜Feedback Control](https://yoshiagent.github.io/automaticControl/FBC-FeedBack.html)、
[Modeling from ODE](https://yoshiagent.github.io/automaticControl/FBC-FeedBack_1.html)、
[PID Examples](https://yoshiagent.github.io/automaticControl/FBC-PID_Examples.html)

原課程用 19 個 Simulink 模型（`.mdl`）示範回授控制。這本筆記本把每個模型的方塊與參數（直接從 `.mdl` 檔讀出，
沒寫的參數用 Simulink 預設值：Gain＝1、Step 在 $t=1$ 由 0 跳到 1）改寫成 Python，並和理論值比對。
受控體大多是質量－彈簧－阻尼器 $m\ddot x+c\dot x+kx=f(t)+d(t)$，$m=c=k=1$：$G(s)=\dfrac{1}{s^2+s+1}$。

| 節 | 原檔 | 內容 |
|---|---|---|
| 1 | `OpenLP1TF.mdl`、`CloseLPTF.mdl` | 開迴路與 P 控制的位置控制（含干擾） |
| 2 | `SensorDis.mdl` | 感測器雜訊：增益越大，雜訊越容易進入輸出 |
| 3 | `CloseLPPITF.mdl`、`PID_IESS.mdl` | PI 控制消除穩態誤差，以及原因 |
| 4 | `CloseLPPITFUStab.mdl`、`PID_Iover.mdl` | 積分增益太大會不穩定 |
| 5 | `PID_P_I_D.mdl`、`PID_P_I_D1.mdl` | P、I、D 三種動作對同一個誤差訊號的反應 |
| 6 | `OpenCloseLPTF.mdl`、`OpenCloseLPPITF.mdl` | 開迴路與閉迴路的抗干擾能力 |
| 7 | `CloseLPPDTF.mdl` | PD 控制：D 提供阻尼 |
| 8 | `OpenCloseLPPITFPV.mdl` | 受控體參數變動 |
| 9 | `OpenCloseLPPITFStab.mdl` | 回授讓不穩定的受控體穩定（PID） |
| 10 | `OpenCloseLP.mdl`、`OpenCloseLPPI.mdl` | 第 6 節的積分器（ODE）版本 |
| 11 | `PID_P.mdl`、`PID_PI.mdl`、`PID_PID.mdl` | PID Examples：$G(s)=\dfrac{1}{s^2+0.2s+1}$ |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def step(t, t0=1.0, after=1.0, before=0.0):
    """Simulink 的 Step 方塊（預設 Step time = 1、Initial value = 0、Final value = 1）"""
    return after if t >= t0 else before


def ramp(t, slope=1.0, start=0.0):
    """Simulink 的 Ramp 方塊"""
    return slope * (t - start) if t >= start else 0.0


class TF:
    """Simulink 的 Transfer Fcn 方塊：num、den 是 s 多項式係數（高次在前，同 MATLAB）。
    內部轉成狀態空間 x' = Ax + Bu、y = Cx + Du；n 是狀態數（＝分母階數）。"""
    def __init__(self, num, den):
        A, B, C, D = signal.tf2ss(num, den)
        self.A, self.B, self.C, self.D = A, B[:, 0], C[0], D[0, 0]
        self.n = len(A)

    def dx(self, x, u):
        return self.A @ x + self.B * u

    def y(self, x, u=0.0):
        return self.C @ x + self.D * u

    def dy(self, x, u=0.0):
        """輸出的微分 y' = C x'（D = 0 時成立；x 可以是整段時間的狀態陣列）"""
        Bu = self.B[:, None] * u if np.ndim(x) == 2 else self.B * u
        return self.C @ (self.A @ x + Bu)


def simulate(rhs, x0, T, breaks=(), dt=0.01, kicks=None, max_step=0.05):
    """仿 Simulink 從 t = 0 跑到 T（ode45 對應 RK45）。
    breaks：輸入跳變的時刻（Step、Ramp 起點），在這些點分段積分，跳變前那一段一律用跳變前的輸入值；
    kicks：{時刻: 狀態增量}，表示 Derivative 方塊遇到跳變輸入時產生的脈衝（見第 9 節）。
    回傳等間隔 dt 的 t 與狀態陣列 X（X[i] 是第 i 個狀態的時間序列）。"""
    n = round(T / dt)
    t = np.linspace(0, T, n + 1)
    cuts = [0.0] + sorted(float(b) for b in breaks if 0 < b < T) + [float(T)]
    x = np.array(x0, dtype=float)
    X = np.empty((len(x), n + 1))
    for a, b in zip(cuts[:-1], cuts[1:]):
        if kicks and a in kicks:
            x = x + np.asarray(kicks[a], dtype=float)
        tb = np.nextafter(b, a)
        sol = solve_ivp(lambda s, z: rhs(min(s, tb), z), (a, b), x, method="RK45",
                        rtol=1e-9, atol=1e-12, max_step=max_step, dense_output=True)
        i0, i1 = round(a / dt), round(b / dt)
        idx = slice(i0, i1 + 1 if b == T else i1)
        X[:, idx] = sol.sol(t[idx])
        x = sol.y[:, -1]
    return t, X


def scope(t, panels, title, h=None, npts=600):
    """像 Simulink 的 Scope：panels = {子圖標題: [(曲線名稱, 數列或 (tx, ty)), …]}。
    數列點數太多時等距抽樣到約 npts 點（只影響畫圖，不影響計算）；(tx, ty) 直接照畫（用於折線形的設定值）。"""
    k = max(1, int(np.ceil(len(t) / npts)))
    fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True, subplot_titles=list(panels), vertical_spacing=0.1)
    for i, traces in enumerate(panels.values(), 1):
        for name, y in traces:
            xs, ys = y if isinstance(y, tuple) else (np.round(t[::k], 3).tolist(), np.round(np.asarray(y)[::k], 4).tolist())
            fig.add_scatter(x=xs, y=ys, name=name, mode="lines", row=i, col=1)
    fig.update_xaxes(title_text="t (s)", row=len(panels), col=1)
    fig.update_layout(title=title, height=h or 260 + 220 * len(panels), template="plotly_white",
                      margin=dict(l=60, r=20, t=70, b=50))
    fig.show()


def sp(t, t0=1.0):
    """設定值（Step：t0 時由 0 跳到 1）的折線端點，畫圖用"""
    return ([0, t0, t0, float(t[-1])], [0, 0, 1, 1])


def val(t, y, tq):
    """y 在 t = tq 的值"""
    return float(np.interp(tq, t, y))


def overshoot(y, final):
    return 100 * (np.max(y) - final) / final


P = TF([1], [1, 1, 1])                                                # 受控體 m = c = k = 1
r = lambda t: step(t)                                                 # Set point：t = 1 由 0 跳到 1
d = lambda t: step(t, 50)                                             # d(t)：t = 50 加入干擾 1

## 1. 位置控制：開迴路與 P 控制 `OpenLP1TF.mdl`、`CloseLPTF.mdl`

* 開迴路（`OpenLP1TF`）：Set Point → Unit Conversion（Gain 1）→ $(+d)$ → $G(s)$。$t=50$ 加入干擾 $d=1$。
* 閉迴路 P 控制（`CloseLPTF`）：$e=r-y$ → $K_p=10$ → $(+d)$ → $G(s)$，$Y=\dfrac{K_pR+D}{s^2+s+1+K_p}$。

理論穩態值：開迴路 $y=1$，加干擾後 $y=2$；P 控制 $y=\dfrac{10}{11}=0.9091$（有穩態誤差），加干擾後 $y=\dfrac{10+1}{11}=1$。

In [2]:
t, X = simulate(lambda t, x: P.dx(x, 1 * r(t) + d(t)), np.zeros(P.n), 100, breaks=(1, 50))
y_open = P.C @ X
print(f"OpenLP1TF : y(49.99) = {val(t, y_open, 49.99):.4f}, y(100) = {val(t, y_open, 100):.4f}")

Kp = 10


def rhs(t, x):
    u = Kp * (r(t) - P.y(x)) + d(t)                                   # Sum4 "|+-" → Kp → Sum3 "++|"（加干擾）
    return P.dx(x, u)


t, X = simulate(rhs, np.zeros(P.n), 100, breaks=(1, 50))
y_p = P.C @ X
print(f"CloseLPTF : y(49.99) = {val(t, y_p, 49.99):.4f}, y(100) = {val(t, y_p, 100):.4f}   (theory 10/11 = {10/11:.4f}, 11/11 = 1)")
scope(t, {"open loop (OpenLP1TF)": [("set point", sp(t)), ("y", y_open)],
          "close loop, P control Kp = 10 (CloseLPTF)": [("set point", sp(t)), ("y", y_p)]},
      "Position control, disturbance d = 1 at t = 50")

OpenLP1TF : y(49.99) = 1.0000, y(100) = 2.0000
CloseLPTF : y(49.99) = 0.9091, y(100) = 1.0000   (theory 10/11 = 0.9091, 11/11 = 1)


## 2. 感測器雜訊 `SensorDis.mdl`

兩個 P 控制迴路，回授訊號加上感測器雜訊 $n(t)=\sin 50t$：$e=r-(y+n)$，增益分別為 $K=1$ 與 $K=500$。
從雜訊到輸出的轉移函數是 $-\dfrac{K}{s^2+s+1+K}$：

* $K=1$：穩態 $y=\dfrac{1}{2}$，雜訊幅度 $\dfrac{1}{|2-2500+50j|}\approx0.0004$；
* $K=500$：穩態 $y=\dfrac{500}{501}\approx0.998$（穩態誤差小），但雜訊幅度 $\dfrac{500}{|501-2500+50j|}\approx0.25$。

In [3]:
noise = lambda t: np.sin(50 * t)                                      # Sine Wave：Frequency 50 rad/s


def sensor_loop(K, nz):
    def rhs(t, x):
        e = r(t) - (P.y(x) + nz(t))                                   # Sum1 = y + 雜訊，Sum "|+-"
        return P.dx(x, K * e)
    t, X = simulate(rhs, np.zeros(P.n), 10, breaks=(1,), dt=0.002, max_step=0.002)
    return t, P.C @ X


panels = {}
for K in (1, 500):
    t, y = sensor_loop(K, noise)
    _, y0 = sensor_loop(K, lambda t: 0.0)                             # 同一迴路不加雜訊；線性系統 → 兩者之差＝雜訊造成的輸出
    m = t >= 8
    a, b = np.linalg.lstsq(np.c_[np.sin(50 * t[m]), np.cos(50 * t[m])], (y - y0)[m], rcond=None)[0]
    th = K / abs(K + 1 - 2500 + 50j)
    print(f"SensorDis K = {K:3d}: noise amplitude in y = {np.hypot(a, b):.4f} (theory {th:.4f}), DC level -> {K / (K + 1):.3f}")
    panels[f"K = {K}"] = [("set point", sp(t)), ("y", y)]
scope(t, panels, "SensorDis.mdl: sensor noise sin(50t) in the feedback path", npts=1700)

SensorDis K =   1: noise amplitude in y = 0.0004 (theory 0.0004), DC level -> 0.500


SensorDis K = 500: noise amplitude in y = 0.2502 (theory 0.2500), DC level -> 0.998


高增益讓穩態誤差變小，但感測器雜訊也幾乎原封不動地出現在輸出：回授只能壓抑「進入受控體」的干擾，
無法分辨感測器量錯的訊號。這是回授設計「穩態誤差」與「雜訊」之間的取捨。

## 3. PI 控制消除穩態誤差 `CloseLPPITF.mdl`、`PID_IESS.mdl`

`CloseLPPITF`：$e$ → $K_P=0.1$ 與 $K_I=0.2$ → 積分器，兩路相加後加上干擾進入 $G(s)$：
$G_c(s)=K_P+\dfrac{K_I}{s}$，特性方程式 $s^3+s^2+(1+K_P)s+K_I=0$。
加了積分器後是 Type 1 系統，步階輸入與步階干擾的穩態誤差都是 0。

In [4]:
def pi_closed(KP, KI, plant=P, dist=d, T=100):
    """PI 控制閉迴路：Sum4 = r − y → KP、KI → Integrator → Sum3 = KP·e + ∫KI·e，再加干擾進入受控體"""
    n = plant.n

    def rhs(t, s):
        x, z = s[:n], s[n]
        e = r(t) - plant.y(x)
        u = dist(t) + KP * e + z
        return np.r_[plant.dx(x, u), KI * e]
    t, X = simulate(rhs, np.zeros(n + 1), T, breaks=(1, 50))
    return t, plant.C @ X[:n]


t, y_pi = pi_closed(0.1, 0.2)
print(f"CloseLPPITF: y(49.99) = {val(t, y_pi, 49.99):.4f}, y(100) = {val(t, y_pi, 100):.4f}")
print("closed-loop poles:", 0 + np.round(np.roots([1, 1, 1.1, 0.2]), 4))
scope(t, {"close loop, PI control KP = 0.1, KI = 0.2": [("set point", sp(t)), ("y", y_pi)]},
      "CloseLPPITF.mdl: no steady-state error, disturbance rejected")

CloseLPPITF: y(49.99) = 1.0000, y(100) = 1.0000
closed-loop poles: [-0.3926+0.8816j -0.3926-0.8816j -0.2147+0.j    ]


**為什麼積分器能消除穩態誤差？**（`PID_IESS`）把一段誤差脈波（$t=1$ 到 $t=5$ 為 1，之後為 0）送進積分器：
誤差回到 0 之後，積分器的輸出仍然停在 4。所以在穩態時，即使 $e=0$，PI 控制器仍能輸出維持位置所需的控制量；
純 P 控制做不到（$e=0$ 時 $u=K_pe=0$）。

In [5]:
pulse = lambda t: step(t, 1) + step(t, 5, after=-1)                   # Step + Step1（t = 5、Final value −1）
t, Z = simulate(lambda t, z: [pulse(t)], [0], 10, breaks=(1, 5))
print(f"PID_IESS: integrator output at t = 10 is {Z[0, -1]:.4f} while the input is 0")
scope(t, {"Mux: input e(t) and integrator output": [("e(t)", ([0, 1, 1, 5, 5, 10], [0, 0, 1, 1, 0, 0])), ("integral of e", Z[0])]},
      "PID_IESS.mdl")

PID_IESS: integrator output at t = 10 is 4.0000 while the input is 0


## 4. 積分增益太大會不穩定 `CloseLPPITFUStab.mdl`、`PID_Iover.mdl`

`CloseLPPITFUStab` 只把 $K_I$ 從 0.2 改成 2：特性方程式 $s^3+s^2+1.1s+2=0$。
Routh 準則：三階 $s^3+a_2s^2+a_1s+a_0$ 穩定的條件是 $a_2a_1>a_0$，這裡 $1\times1.1<2$，不穩定。

In [6]:
t, y_us = pi_closed(0.1, 2.0)
poles = np.roots([1, 1, 1.1, 2])
print("CloseLPPITFUStab poles:", np.round(poles, 4), f"-> max Re = {np.round(poles.real.max(), 4) + 0.0:+.4f} (unstable)")
print(f"Routh: a2*a1 - a0 = {1 * 1.1 - 2:.1f} < 0")
scope(t, {"close loop, KP = 0.1, KI = 2": [("set point", sp(t)), ("y", y_us)]},
      "CloseLPPITFUStab.mdl: oscillation grows without bound")

CloseLPPITFUStab poles: [-1.3174+0.j      0.1587+1.2219j  0.1587-1.2219j] -> max Re = +0.1587 (unstable)
Routh: a2*a1 - a0 = -0.9 < 0


`PID_Iover`：只用積分控制器（$K_I=1$）。閉迴路 $\dfrac{Y}{R}=\dfrac{1}{s(s^2+s+1)+1}=\dfrac{1}{(s+1)(s^2+1)}$，
一對極點正好在虛軸上 $\pm j$：響應永遠振盪，$y=1-\tfrac12e^{-\tau}-\tfrac12\cos\tau-\tfrac12\sin\tau$（$\tau=t-1$）。
原因：積分器要等誤差累積夠久才有足夠輸出，等到 $y$ 衝過頭、誤差變號，積分值還要一段時間才降下來，造成過衝與振盪。

In [7]:
def rhs(t, s):
    x, z = s[:2], s[2]
    e = r(t) - P.y(x)
    return np.r_[P.dx(x, z), e]                                       # I Controller：u = ∫e


t, X = simulate(rhs, np.zeros(3), 10, breaks=(1,))
y_i, u_i = P.C @ X[:2], X[2]
e_i = np.where(t >= 1, 1.0, 0.0) - y_i
tau = np.clip(t - 1, 0, None)
y_exact = np.where(t >= 1, 1 - 0.5 * np.exp(-tau) - 0.5 * np.cos(tau) - 0.5 * np.sin(tau), 0)
print("PID_Iover poles:", 0 + np.round(np.roots([1, 1, 1, 1]), 4))
print(f"PID_Iover: max |y - analytic| = {np.abs(y_i - y_exact).max():.1e}")
scope(t, {"Comparison (Mux): error, integral output, position": [("error e", e_i), ("integral output u", u_i), ("position y", y_i)]},
      "PID_Iover.mdl: pure I control of 1/(s^2+s+1)")

PID_Iover poles: [-1.+0.j  0.+1.j  0.-1.j]
PID_Iover: max |y - analytic| = 4.1e-10


## 5. P、I、D 三種動作 `PID_P_I_D.mdl`、`PID_P_I_D1.mdl`

這兩個模型沒有回授，只是把同一個誤差訊號 $e(t)$ 分別送進 P（$e$ 本身）、I（$\int e\,dt$）、D（$\dot e$）看反應：

* `PID_P_I_D`：Step（$t=0$）→ $G(s)$ → $y$；$e=1-y$。Mux 四條線：$\int e$、$e$、$-\dot y$（Derivative → Gain $-1$，因為 $r$ 固定時 $\dot e=-\dot y$）、$y$。
* `PID_P_I_D1`：輸入改成梯形 $r(t)$（Ramp 斜率 1/5 減去 $t=5$ 開始的 Ramp，$t\ge5$ 時停在 1）；$e=r-y$，D 動作 $5\dot e$。

In [8]:
def rhs(t, s):
    x = s[:2]
    e = 1.0 - P.y(x)
    return np.r_[P.dx(x, 1.0), e]


t, X = simulate(rhs, np.zeros(3), 10)
y = P.C @ X[:2]
e, I_act, D_act = 1 - y, X[2], -P.dy(X[:2], 1.0)
print(f"PID_P_I_D at t = 10: P = e = {e[-1]:.4f}, I = {I_act[-1]:.4f}, D = -y' = {D_act[-1]:.4f}")
scope(t, {"Mux: I, P, D actions and output": [("I: integral of e", I_act), ("P: e", e), ("D: -y'", D_act), ("y", y)]},
      "PID_P_I_D.mdl: step into 1/(s^2+s+1), e = 1 - y")

rr = lambda t: ramp(t, 0.2, 0) - ramp(t, 0.2, 5)                      # Ramp − Ramp1（start = 5）
t, X = simulate(lambda t, x: P.dx(x, rr(t)), np.zeros(2), 10, breaks=(5,))
y = P.C @ X
rv = np.array([rr(s) for s in t])
e = rv - y                                                            # Sum1 "-+|"
D_act = 5 * (np.where(t < 5, 0.2, 0.0) - P.dy(X, rv))                 # Derivative → Gain 5
print(f"PID_P_I_D1: D action 5e' = {D_act[0]:.4f} at t = 0, {val(t, D_act, 4.99):.4f} just before t = 5, {D_act[t >= 5][0]:.4f} just after")
scope(t, {"Mux: r, y, 5 de/dt, e": [("r", rv), ("y", y), ("D: 5 de/dt", D_act), ("e", e)]},
      "PID_P_I_D1.mdl: trapezoid input, e = r - y")

PID_P_I_D at t = 10: P = e = -0.0022, I = 1.0076, D = -y' = -0.0054


PID_P_I_D1: D action 5e' = 1.0000 at t = 0, -0.0755 just before t = 5, -1.0746 just after


網頁的結論：積分控制器在響應前期累積得慢（甚至造成過衝），但後期能消除穩態誤差；
微分控制器對誤差的「變化」反應，誤差還沒變大就先動作，有阻尼（預測）作用，對穩態幾乎沒有影響。
（Simulink 的 Derivative 方塊在 $t=0$ 的輸出固定為 0，上面算的是 $t=0^+$ 的值。）

## 6. 開迴路與閉迴路的抗干擾 `OpenCloseLPTF.mdl`、`OpenCloseLPPITF.mdl`

同一個受控體並排比較：開迴路直接送入設定值（Gain5＝1），閉迴路用 P（$K_p=10$）或 PI（$K_P=0.1$、$K_I=0.2$）控制。
`OpenCloseLPTF` 的干擾在 $t=50$ 加入 2，`OpenCloseLPPITF` 的干擾加入 1。

理論：P 控制 $y=\dfrac{10}{11}\to\dfrac{10+2}{11}=1.0909$，開迴路 $1\to3$；PI 控制兩段都是 1，開迴路 $1\to2$。

In [9]:
d2 = lambda t: step(t, 50, after=2)                                   # disturbance：Final value 2
t, X = simulate(lambda t, x: P.dx(x, r(t) + d2(t)), np.zeros(2), 100, breaks=(1, 50))
y_open2 = P.C @ X
t, X = simulate(lambda t, x: P.dx(x, d2(t) + 10 * (r(t) - P.y(x))), np.zeros(2), 100, breaks=(1, 50))
y_p2 = P.C @ X
print(f"OpenCloseLPTF  : open  y(49.99) = {val(t, y_open2, 49.99):.4f}, y(100) = {val(t, y_open2, 100):.4f}")
print(f"                 close y(49.99) = {val(t, y_p2, 49.99):.4f}, y(100) = {val(t, y_p2, 100):.4f}")
scope(t, {"P control, d = 2 at t = 50": [("open loop", y_open2), ("close loop (Kp = 10)", y_p2)]}, "OpenCloseLPTF.mdl")

t, X = simulate(lambda t, x: P.dx(x, r(t) + d(t)), np.zeros(2), 100, breaks=(1, 50))
y_open1 = P.C @ X
t, y_pi1 = pi_closed(0.1, 0.2)
print(f"OpenCloseLPPITF: open  y(49.99) = {val(t, y_open1, 49.99):.4f}, y(100) = {val(t, y_open1, 100):.4f}")
print(f"                 close y(49.99) = {val(t, y_pi1, 49.99):.4f}, y(100) = {val(t, y_pi1, 100):.4f}")
scope(t, {"PI control, d = 1 at t = 50": [("open loop", y_open1), ("close loop (KP = 0.1, KI = 0.2)", y_pi1)]}, "OpenCloseLPPITF.mdl")

OpenCloseLPTF  : open  y(49.99) = 1.0000, y(100) = 3.0000
                 close y(49.99) = 0.9091, y(100) = 1.0909


OpenCloseLPPITF: open  y(49.99) = 1.0000, y(100) = 2.0000
                 close y(49.99) = 1.0000, y(100) = 1.0000


開迴路完全不知道干擾的存在，輸出跟著干擾跑；P 控制把干擾的影響縮小成 $\dfrac{1}{1+K_p}$；PI 控制則完全消除。

## 7. PD 控制 `CloseLPPDTF.mdl`

受控體 $\ddot x+0.2\dot x+x$（Gain c＝0.2、k＝1，兩個積分器），P 控制 $K_P=1$，D 項取速度回授 $K_D\dot x$：
$\ddot x+(0.2+K_D)\dot x+(1+K_P)x=K_Pu_s(t)$。模型並排兩組：$K_D=0$ 與 $K_D=1$。
阻尼比 $\zeta=\dfrac{0.2+K_D}{2\sqrt{1+K_P}}$：0.071 → 0.424；穩態值都是 $\dfrac{K_P}{1+K_P}=0.5$。

In [10]:
def pd_loop(KP, KD):
    def rhs(t, s):
        x, v = s
        acc = -1 * x + (KP * (r(t) - x) - KD * v) - 0.2 * v          # Sum2 "-+-"：−k·x、Sum = KP·e − KD·x'、−c·x'
        return [v, acc]
    t, X = simulate(rhs, [0, 0], 100, breaks=(1,))
    return t, X[0]


panels = {}
for KD in (0, 1):
    t, x = pd_loop(1, KD)
    zeta = (0.2 + KD) / (2 * np.sqrt(2))
    Mp = 100 * np.exp(-zeta * np.pi / np.sqrt(1 - zeta**2))
    print(f"CloseLPPDTF KD = {KD}: zeta = {zeta:.3f}, overshoot = {overshoot(x, 0.5):.1f} % (theory {Mp:.1f} %), x(100) = {x[-1]:.4f}")
    panels[f"KD = {KD}"] = [("set point", sp(t)), ("x", x)]
scope(t, panels, "CloseLPPDTF.mdl: derivative action adds damping")

CloseLPPDTF KD = 0: zeta = 0.071, overshoot = 80.0 % (theory 80.0 %), x(100) = 0.5000
CloseLPPDTF KD = 1: zeta = 0.424, overshoot = 22.9 % (theory 22.9 %), x(100) = 0.5000


## 8. 受控體參數變動 `OpenCloseLPPITFPV.mdl`

兩個受控體 $\dfrac{1}{s^2+s+1}$ 與 $\dfrac{1}{s^2+s+2}$（彈簧常數由 1 變成 2）。開迴路的穩態值跟著變（1 與 0.5），
同一組 PI 控制器（$K_P=0.1$、$K_I=0.2$）的閉迴路穩態值都是 1：回授降低系統對參數變動的敏感度。

In [11]:
P2 = TF([1], [1, 1, 2])
panels = {"open loop": [], "close loop (PI)": []}
for name, plant in [("1/(s^2+s+1)", P), ("1/(s^2+s+2)", P2)]:
    t, X = simulate(lambda t, x: plant.dx(x, r(t)), np.zeros(2), 100, breaks=(1,))
    yo = plant.C @ X
    t, yc = pi_closed(0.1, 0.2, plant=plant, dist=lambda t: 0.0)
    print(f"OpenCloseLPPITFPV {name}: open y(100) = {yo[-1]:.4f}, close y(100) = {yc[-1]:.3f}")
    panels["open loop"].append((name, yo))
    panels["close loop (PI)"].append((name, yc))
scope(t, panels, "OpenCloseLPPITFPV.mdl: plant parameter variation")

OpenCloseLPPITFPV 1/(s^2+s+1): open y(100) = 1.0000, close y(100) = 1.000


OpenCloseLPPITFPV 1/(s^2+s+2): open y(100) = 0.5000, close y(100) = 1.000


## 9. 回授讓不穩定的受控體穩定 `OpenCloseLPPITFStab.mdl`

受控體 $\dfrac{1}{s^2-0.1s+1}$（負阻尼，極點 $0.05\pm j0.999$ 在右半平面），開迴路的振盪越來越大。
閉迴路 PID：$K_P=30$、$K_I=60$，D 項是 $K_D=5$ → Derivative 方塊，作用在誤差 $e=r-y$ 上。
特性方程式 $s^3+4.9s^2+31s+60=0$，Routh：$4.9\times31>60$，穩定。

**Derivative 方塊與步階的處理**：設定值在 $t=1$ 由 0 跳到 1，$\dot e$ 在那一瞬間是脈衝 $\delta(t-1)$，
所以 D 項送給受控體一個大小 $K_D$ 的脈衝（Simulink 以 MaxStep 0.01 的差分近似它）。Python 這邊用
`kicks`：在 $t=1$ 讓受控體狀態瞬間增加 $B\cdot K_D$；其餘時間 $\dot r=0$，D 項就是 $-K_D\dot y$。
下面再用閉迴路轉移函數 $\dfrac{K_Ds^2+K_Ps+K_I}{s^3+(K_D-0.1)s^2+(1+K_P)s+K_I}$ 的步階響應驗算。

In [12]:
Pu = TF([1], [1, -0.1, 1])
t, X = simulate(lambda t, x: Pu.dx(x, 1 * r(t)), np.zeros(2), 50, breaks=(1,))
y_ol = Pu.C @ X
print("open-loop poles:", 0 + np.round(np.roots([1, -0.1, 1]), 4))

KP, KI, KD = 30, 60, 5


def rhs(t, s):
    x, z = s[:2], s[2]
    e = r(t) - Pu.y(x)
    u = KP * e + z - KD * Pu.dy(x)                                    # D 項 KD·(r' − y')，t ≠ 1 時 r' = 0
    return np.r_[Pu.dx(x, u), KI * e]


t, X = simulate(rhs, np.zeros(3), 50, breaks=(1,), kicks={1.0: np.r_[Pu.B * KD, 0.0]})
y_cl = Pu.C @ X[:2]
num, den = [KD, KP, KI], [1, KD - 0.1, 1 + KP, KI]
y_tf = np.zeros_like(t)
m = t >= 1
_, y_tf[m] = signal.step((num, den), T=t[m] - 1)
print("closed-loop poles:", 0 + np.round(np.roots(den), 4))
print(f"OpenCloseLPPITFStab: close y(50) = {y_cl[-1]:.4f}; max |simulation - closed-loop TF| = {np.abs(y_cl - y_tf).max():.1e}")
scope(t, {"Open Loop: 1/(s^2-0.1s+1)": [("y", y_ol)],
          "Close Loop: PID KP = 30, KI = 60, KD = 5": [("set point", sp(t)), ("y", y_cl)]},
      "OpenCloseLPPITFStab.mdl: feedback stabilizes an unstable plant")

open-loop poles: [0.05+0.9987j 0.05-0.9987j]
closed-loop poles: [-1.25+4.8412j -1.25-4.8412j -2.4 +0.j    ]
OpenCloseLPPITFStab: close y(50) = 1.0000; max |simulation - closed-loop TF| = 3.1e-10


## 10. 用積分器建模（Modeling from ODE）`OpenCloseLP.mdl`、`OpenCloseLPPI.mdl`

這兩個模型和第 6 節相同，只是受控體不用 Transfer Fcn，而是照 $\ddot x=f(t)+d(t)-\dot x-x$ 用兩個積分器搭出來
（Sum 方塊 `-+-`：$-kx$、$+(f+d)$、$-c\dot x$）。`OpenCloseLP` 是 P 控制（$K_p=10$，干擾 2），
`OpenCloseLPPI` 是 PI 控制（$K_P=0.1$、$K_I=0.2$，干擾 1）。結果應與第 6 節完全一樣。

In [13]:
def rhs(t, s):
    xo, vo, xc, vc = s
    ao = -xo + (1 * r(t) + d2(t)) - vo                                # 開迴路：Sum "-+-"，Sum1 = Gain5·r + d
    ac = -xc + (d2(t) + 10 * (r(t) - xc)) - vc                        # 閉迴路：Sum3 = d + Gain4·(r − x)
    return [vo, ao, vc, ac]


t, X = simulate(rhs, np.zeros(4), 100, breaks=(1, 50))
print(f"OpenCloseLP  : max difference from OpenCloseLPTF   = {max(np.abs(X[0] - y_open2).max(), np.abs(X[2] - y_p2).max()):.1e}")


def rhs(t, s):
    xo, vo, xc, vc, z = s
    e = r(t) - xc
    ao = -xo + (1 * r(t) + d(t)) - vo
    ac = -xc + (d(t) + 0.1 * e + z) - vc                              # Sum3 "+++"：d + KP·e + ∫KI·e
    return [vo, ao, vc, ac, 0.2 * e]


t, X = simulate(rhs, np.zeros(5), 100, breaks=(1, 50))
print(f"OpenCloseLPPI: max difference from OpenCloseLPPITF = {max(np.abs(X[0] - y_open1).max(), np.abs(X[2] - y_pi1).max()):.1e}")
scope(t, {"open loop (integrator model)": [("x", X[0])], "close loop, PI (integrator model)": [("x", X[2])]},
      "OpenCloseLPPI.mdl")

OpenCloseLP  : max difference from OpenCloseLPTF   = 5.2e-10
OpenCloseLPPI: max difference from OpenCloseLPPITF = 7.8e-11


## 11. PID Examples `PID_P.mdl`、`PID_PI.mdl`、`PID_PID.mdl`

受控體 $G(s)=\dfrac{1}{s^2+0.2s+1}$（$\omega_n=1$、$\zeta=0.1$）。

* `PID_P`：$K_p=1$ 的迴路，和直接用閉迴路轉移函數 $\dfrac{1}{s^2+0.2s+2}$ 的方塊並排，兩者相同，穩態值 0.5。
* `PID_PI`：$K_p=3$、$K_I=0.3$，Step 在 $t=0$，模擬 200 秒，Display 顯示最後的輸出。
* `PID_PID`：$K_p=16$、$K_I=10$、$K_d=8$，Step 在 $t=0$，模擬 50 秒。

注意 `PID_PID` 的 Step time 是 0：Simulink 的 Step 在 $t=0$ 已經是 1，Derivative 方塊的初始輸出是 0，
所以 D 項**看不到**設定值的跳變，實際上等於 $-K_d\dot y$（不會有第 9 節的脈衝）。
下面另外畫出「有脈衝」的理想 PID（閉迴路轉移函數）做比較。

In [14]:
Pd = TF([1], [1, 0.2, 1])
Pcl = TF([1], [1, 0.2, 2])                                            # Closed Loop Transfer Fcn 方塊
t, X = simulate(lambda t, s: np.r_[Pd.dx(s[:2], 1 * (r(t) - Pd.y(s[:2]))), Pcl.dx(s[2:], r(t))], np.zeros(4), 50, breaks=(1,))
y_loop, y_cltf = Pd.C @ X[:2], Pcl.C @ X[2:]
print(f"PID_P  : max |loop - closed-loop TF block| = {np.abs(y_loop - y_cltf).max():.1e}, overshoot = {overshoot(y_loop, 0.5):.1f} %")
scope(t, {"Scope: Kp = 1 loop": [("y", y_loop)], "Scope1: 1/(s^2+0.2s+2)": [("y", y_cltf)]}, "PID_P.mdl")


def pid(Kp, KI, Kd, T):
    def rhs(t, s):
        x, z = s[:2], s[2]
        e = 1.0 - Pd.y(x)                                             # Step time = 0
        u = Kp * e + z - Kd * Pd.dy(x)
        return np.r_[Pd.dx(x, u), KI * e]
    return simulate(rhs, np.zeros(3), T)


t, X = pid(3, 0.3, 0, 200)
y_pi = Pd.C @ X[:2]
print(f"PID_PI : Display y(200) = {y_pi[-1]:.4f}; poles", 0 + np.round(np.roots([1, 0.2, 4, 0.3]), 4))
scope(t, {"Kp = 3, KI = 0.3": [("y", y_pi)]}, "PID_PI.mdl")

t, X = pid(16, 10, 8, 50)
y_pid = Pd.C @ X[:2]
_, y_ideal = signal.step(([8, 16, 10], [1, 8.2, 17, 10]), T=t)
print(f"PID_PID: Display y(50) = {y_pid[-1]:.4f}, overshoot = {overshoot(y_pid, 1):.1f} % (Simulink, no derivative kick);"
      f" ideal PID with kick: overshoot = {overshoot(y_ideal, 1):.1f} %")
scope(t, {"Kp = 16, KI = 10, Kd = 8": [("Simulink model (step at t = 0)", y_pid), ("ideal PID transfer function", y_ideal)]},
      "PID_PID.mdl", h=420)

PID_P  : max |loop - closed-loop TF block| = 4.4e-16, overshoot = 80.0 %

PID_PI : Display y(200) = 1.0000; poles [-0.0624+1.9967j -0.0624-1.9967j -0.0752+0.j    ]


PID_PID: Display y(50) = 1.0000, overshoot = 15.4 % (Simulink, no derivative kick); ideal PID with kick: overshoot = 11.5 %


三個例子對照：P 控制有穩態誤差（0.5），而且 $\zeta$ 只有 0.07，振盪很久；PI 消除了穩態誤差，但增益小、反應慢（要 200 秒）；
PID 加入微分項提供阻尼，在 50 秒內就快速而平穩地到達 1。